In [0]:
%sql
CREATE OR REPLACE TABLE workspace.us_electricity.gold_energy_inflation_monthly
USING DELTA
AS

WITH valid_observations AS (

    SELECT
        series_id,
        observation_date,
        value

    FROM workspace.us_electricity.silver_fred_observations

    WHERE is_valid = TRUE

),

-- Convert the long FRED structure into one row per month
monthly AS (

    SELECT
        observation_date AS month,

        MAX(
            CASE
                WHEN series_id = 'APU000072610'
                THEN value
            END
        ) AS electricity_price_per_kwh_usd,

        MAX(
            CASE
                WHEN series_id = 'CUSR0000SEHF01'
                THEN value
            END
        ) AS electricity_cpi,

        MAX(
            CASE
                WHEN series_id = 'CPIAUCSL'
                THEN value
            END
        ) AS headline_cpi,

        MAX(
            CASE
                WHEN series_id = 'CPILFESL'
                THEN value
            END
        ) AS core_cpi,

        MAX(
            CASE
                WHEN series_id = 'MHHNGSP'
                THEN value
            END
        ) AS natural_gas_price_per_mmbtu

    FROM valid_observations

    GROUP BY observation_date

),

-- Pull the value from 12 months earlier for YoY calculations
prior_year AS (

    SELECT
        *,

        LAG(electricity_cpi, 12)
            OVER (ORDER BY month)
            AS electricity_cpi_12m_ago,

        LAG(headline_cpi, 12)
            OVER (ORDER BY month)
            AS headline_cpi_12m_ago,

        LAG(core_cpi, 12)
            OVER (ORDER BY month)
            AS core_cpi_12m_ago,

        LAG(natural_gas_price_per_mmbtu, 12)
            OVER (ORDER BY month)
            AS natural_gas_price_12m_ago

    FROM monthly

),

-- Convert the indexes into analytically useful metrics
metrics AS (

    SELECT
        month,

        electricity_price_per_kwh_usd,
        electricity_cpi,
        headline_cpi,
        core_cpi,
        natural_gas_price_per_mmbtu,

        100 * (
            electricity_cpi / electricity_cpi_12m_ago - 1
        ) AS electricity_inflation_yoy_pct,

        100 * (
            headline_cpi / headline_cpi_12m_ago - 1
        ) AS headline_inflation_yoy_pct,

        100 * (
            core_cpi / core_cpi_12m_ago - 1
        ) AS core_inflation_yoy_pct,

        100 * (
            natural_gas_price_per_mmbtu
            / natural_gas_price_12m_ago - 1
        ) AS natural_gas_price_yoy_pct

    FROM prior_year

),

-- Calculate differences between electricity inflation
-- and broader measures of inflation
comparisons AS (

    SELECT
        *,

        electricity_inflation_yoy_pct
            - headline_inflation_yoy_pct
            AS electricity_vs_headline_spread_pp,

        electricity_inflation_yoy_pct
            - core_inflation_yoy_pct
            AS electricity_vs_core_spread_pp

    FROM metrics

),

-- Add rolling relationship measures
rolling_metrics AS (

    SELECT
        *,

        CASE
            WHEN COUNT(electricity_inflation_yoy_pct)
                OVER (
                    ORDER BY month
                    ROWS BETWEEN 23 PRECEDING AND CURRENT ROW
                ) = 24

            THEN CORR(
                electricity_inflation_yoy_pct,
                headline_inflation_yoy_pct
            ) OVER (
                ORDER BY month
                ROWS BETWEEN 23 PRECEDING AND CURRENT ROW
            )

        END AS electricity_headline_corr_24m

    FROM comparisons

)

SELECT
    month,

    electricity_price_per_kwh_usd,

    electricity_cpi,
    headline_cpi,
    core_cpi,

    natural_gas_price_per_mmbtu,

    electricity_inflation_yoy_pct,
    headline_inflation_yoy_pct,
    core_inflation_yoy_pct,

    natural_gas_price_yoy_pct,

    electricity_vs_headline_spread_pp,
    electricity_vs_core_spread_pp,

    electricity_headline_corr_24m,

    CURRENT_TIMESTAMP() AS processed_at

FROM rolling_metrics

ORDER BY month;